In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC-Taxi-Batch") \
    .getOrCreate()

df = spark.read.csv(
    "train.csv",
    header=True,
    inferSchema=True
)

In [9]:
# Nombre de colonnes
print(len(df.columns))

# Types de données de chaque colonne
df.printSchema()


11
root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)



In [10]:
# Nombre de colonnes
n_cols = len(df.columns)
print(f"Colonnes : {n_cols}")

# Nombre de lignes 
n_rows = df.count()
print(f"Lignes : {n_rows}")

Colonnes : 11
Lignes : 1458644


In [11]:
from pyspark.sql import functions as F

# calcul temps en minutes -> nouvelle colonnes
df = df.withColumn("trip_duration_in_minutes", F.col("trip_duration") / 60)

# rennomage de colonnes
df = df.withColumnRenamed("pickup_datetime", "pickup_time") \
       .withColumnRenamed("dropoff_datetime", "dropoff_time")

# suppresion de colonne
df = df.drop("store_and_fwd_flag")

df.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- trip_duration_in_minutes: double (nullable = true)



In [12]:
# filtrage
df_filtered = df.filter((F.col("passenger_count") > 2) & (F.col("trip_duration") < 600))

# cast en string + tri décroissant
df_filtered = df_filtered.withColumn("vendor_id", F.col("vendor_id").cast("string"))
df_filtered = df_filtered.orderBy(F.col("trip_duration_in_minutes").desc())

df_filtered.printSchema()
df_filtered.show(5)

root
 |-- id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- trip_duration_in_minutes: double (nullable = true)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+-------------+------------------------+
|       id|vendor_id|        pickup_time|       dropoff_time|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|trip_duration|trip_duration_in_minutes|
+---------+---------+-------------------+-------------------+---------------+------------------+--

In [13]:
# union des deux df

df_union = df_filtered.union(df)
df_union.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- trip_duration_in_minutes: double (nullable = true)



In [15]:
# nouvelle colonne date du trip
df = df.withColumn("trip_date", F.to_date(F.col("pickup_time")))
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- trip_duration_in_minutes: double (nullable = true)
 |-- trip_date: date (nullable = true)



In [18]:
# moyenne des durée des trajets par jour
avg_duration_by_day = df.groupBy("trip_date") \
    .agg(F.avg("trip_duration_in_minutes").alias("avg_trip_duration_minutes")) \
    .orderBy("trip_date")

avg_duration_by_day.show(10)
avg_duration_by_day.count()

AttributeError: 'GroupedData' object has no attribute 'show'

In [20]:
test = df.groupBy("trip_date").agg(
    F.avg("trip_duration_in_minutes").alias("avg_duration"),
    F.count("*").alias("nb_trajets"),
    F.min("trip_duration_in_minutes").alias("min_duration"),
    F.max("trip_duration_in_minutes").alias("max_duration"),
    F.sum("passenger_count").alias("total_passengers")
)

test.show(10)
test.count()

+----------+------------------+----------+--------------------+------------------+----------------+
| trip_date|      avg_duration|nb_trajets|        min_duration|      max_duration|total_passengers|
+----------+------------------+----------+--------------------+------------------+----------------+
|2016-03-01| 14.96489348131139|      7839| 0.03333333333333333|           1433.25|           12814|
|2016-04-25|14.142572566875353|      7028|                0.05|1433.0666666666666|           11621|
|2016-05-03| 17.23398171140278|      8457|                0.05|1439.1833333333334|           13931|
|2016-01-28| 17.05130589304901|      8066| 0.03333333333333333|           1436.35|           13293|
|2016-06-02| 18.08571371371371|      8325| 0.03333333333333333|1439.1666666666667|           13710|
|2016-02-04|15.703515578369348|      8377|                0.05|1438.5333333333333|           13475|
|2016-05-26| 17.47198005171777|      8121|                0.05|1439.4833333333333|           13132|


182

In [21]:
# nombre de partition
print(df.rdd.getNumPartitions())

32


In [23]:
import time

# SANS repartitionnement (baseline)
t0 = time.time()
df.groupBy("trip_date").agg(F.avg("trip_duration_in_minutes").alias("avg_duration")).count()
print(f"Sans repartition : {time.time() - t0:.2f}s")

# 2. AVEC repartition() sur trip_date
df_repartitioned = df.repartition("trip_date")
t0 = time.time()
df_repartitioned.groupBy("trip_date").agg(F.avg("trip_duration_in_minutes").alias("avg_duration")).count()
print(f"Avec repartition('trip_date') : {time.time() - t0:.2f}s")
print(f"Nb partitions après repartition : {df_repartitioned.rdd.getNumPartitions()}")

# 3. AVEC coalesce()  4 partitions
df_coalesced = df.coalesce(4)
t0 = time.time()
df_coalesced.groupBy("trip_date").agg(F.avg("trip_duration_in_minutes").alias("avg_duration")).count()
print(f"Avec coalesce(4) : {time.time() - t0:.2f}s")
print(f"Nb partitions après coalesce : {df_coalesced.rdd.getNumPartitions()}")

Sans repartition : 0.39s
Avec repartition('trip_date') : 0.63s
Nb partitions après repartition : 40
Avec coalesce(4) : 0.57s
Nb partitions après coalesce : 4


In [24]:
# bloc sans cache

avg_duration_by_day = df.groupBy("trip_date") \
    .agg(F.avg("trip_duration_in_minutes").alias("avg_duration"))

# calcul complet depuis le CSV
t0 = time.time()
avg_duration_by_day.count()
print(f"1er accès (sans cache) : {time.time() - t0:.2f}s")

# recalcule tout, encore
t0 = time.time()
avg_duration_by_day.count()
print(f"2e accès (sans cache) : {time.time() - t0:.2f}s")

# recalcule tout, encore une fois
t0 = time.time()
avg_duration_by_day.count()
print(f"3e accès (sans cache) : {time.time() - t0:.2f}s")

1er accès (sans cache) : 0.64s
2e accès (sans cache) : 0.41s
3e accès (sans cache) : 0.38s


In [26]:
# avec cache

avg_duration_by_day_cached = df.groupBy("trip_date") \
    .agg(F.avg("trip_duration_in_minutes").alias("avg_duration"))

avg_duration_by_day_cached.cache()

# calcul complet + écriture en cache 
t0 = time.time()
avg_duration_by_day_cached.count()
print(f"1er accès (calcul + mise en cache) : {time.time() - t0:.2f}s")

#lu depuis le cache, pas recalculé
t0 = time.time()
avg_duration_by_day_cached.count()
print(f"2e accès (depuis le cache) : {time.time() - t0:.2f}s")

# pareil
t0 = time.time()
avg_duration_by_day_cached.count()
print(f"3e accès (depuis le cache) : {time.time() - t0:.2f}s")

1er accès (calcul + mise en cache) : 0.03s
2e accès (depuis le cache) : 0.02s
3e accès (depuis le cache) : 0.02s


In [27]:
trips_by_day_vendor = df.groupBy("trip_date", "vendor_id") \
    .agg(F.count("*").alias("nb_trajets")) \
    .orderBy("trip_date", "vendor_id")

trips_by_day_vendor.show(20)

+----------+---------+----------+
| trip_date|vendor_id|nb_trajets|
+----------+---------+----------+
|2016-01-01|        1|      3068|
|2016-01-01|        2|      4094|
|2016-01-02|        1|      2905|
|2016-01-02|        2|      3607|
|2016-01-03|        1|      2916|
|2016-01-03|        2|      3437|
|2016-01-04|        1|      3178|
|2016-01-04|        2|      3547|
|2016-01-05|        1|      3322|
|2016-01-05|        2|      3882|
|2016-01-06|        1|      3420|
|2016-01-06|        2|      3945|
|2016-01-07|        1|      3621|
|2016-01-07|        2|      4028|
|2016-01-08|        1|      3843|
|2016-01-08|        2|      4386|
|2016-01-09|        1|      3935|
|2016-01-09|        2|      4643|
|2016-01-10|        1|      3399|
|2016-01-10|        2|      4055|
+----------+---------+----------+
only showing top 20 rows



In [28]:
avg_duration_by_day = df.groupBy("trip_date") \
    .agg(F.avg("trip_duration_in_minutes").alias("avg_duration"))

avg_duration_by_day.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [trip_date#249, avg_duration#1003]
      +- InMemoryRelation [trip_date#249, avg_duration#1003], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=false
               +- HashAggregate(keys=[trip_date#249], functions=[avg(trip_duration_in_minutes#83)])
                  +- Exchange hashpartitioning(trip_date#249, 200), ENSURE_REQUIREMENTS, [plan_id=1395]
                     +- HashAggregate(keys=[trip_date#249], functions=[partial_avg(trip_duration_in_minutes#83)])
                        +- Project [(cast(trip_duration#55 as double) / 60.0) AS trip_duration_in_minutes#83, cast(pickup_datetime#47 as date) AS trip_date#249]
                           +- FileScan csv [pickup_datetime#47,trip_duration#55] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/train.csv], PartitionFilters: [], PushedFilters: [], Rea

In [ ]:


#En affichant .explain(mode="extended"), on voit que Catalyst applique pas mal d'optimisations automatiquement. 
#D'abord, seul les colonnes vraiment utilisées (pickup_datetime, trip_duration) sont lu depuis le CSV, alors qu'il en contient onze — c'est ce qu'on appelle le column pruning. 
#Ensuite, les transformations withColumn sont fusionnées en une seule étape Project. 
#L'optimisation la plus intéressante concerne l'agrégation : Spark calcule d'abord une moyenne partielle localement dans chaque partition, avant même le shuffle, puis combine ces résultats partiels après coup — ça permet de limiter fortement le volume de donnée transféré sur le réseau. 
#On voit aussi apparaître un AdaptiveSparkPlan, qui montre que Spark peut encore ajuster son plan pendant l'exécution, selon les statistique réelles observées sur les données.

#Le plan change bien dans les deux cas qu'on a testé. 
#Avec .repartition("trip_date"), un Exchange (shuffle) supplémentaire apparait : le repartitionnement explicite ne remplace pas celui du groupBy, il vient plutôt s'y ajouter, ce qui explique le temps d'exécution plus élevé qu'on avait observé.
#Avec .cache() par contre, dès le premier accès passé, tout le calcul (lecture, transformation, agrégation) est remplacé par un simple InMemoryTableScan : Spark se rend compte que le résultat est déjà en mémoire et évite donc de tout recalculer depuis le début.